In [1]:
import polars as pl
import pandas as pd

In [2]:
file_path = 'C:/Users/APIN PC/OneDrive/Documents/DS/DE_Inter/aws_project/data/Pakistan Largest Ecommerce Dataset.csv'

cols = ['item_id', 'status', 'created_at', 'sku', 
        'price', 'qty_ordered', 'grand_total', 'increment_id', 
        'category_name_1', 'sales_commission_code', 'discount_amount',
        'payment_method', 'Working Date', 'BI Status', ' MV ', 'Year',
        'Month', 'Customer Since', 'M-Y', 'FY', 'Customer ID']

In [3]:
df_pd = pd.read_csv(file_path, usecols=cols)
df_pl = pl.read_csv(file_path, columns=cols, low_memory=True, ignore_errors=True)

C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_15268\2096382363.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pd = pd.read_csv(file_path, usecols=cols)


In [4]:
def datatypes_comparison():
    # Converting polars read to series
    b = df_pl.dtypes
    pl_series = pd.Series(b)

    # Converting pandas csv read to series and stripping irrelevant data.
    a = df_pd.dtypes
    pd_series = pd.Series(a)
    pl_cols_list = []
    for row in pd_series:
        row.str.strip(' ')
        pl_cols_list.append(str(row))
    pd_series_final = pd.Series(pl_cols_list)

    # Converting columns names to series.
    cols = df_pd.columns
    cols_fin = pd.Series(cols)

    # Concating all series to a dataframe.
    fin_df = pd.concat([cols_fin, pd_series_final, pl_series], axis=1)
    return fin_df

In [ ]:
dtypes_df = datatypes_comparison()

dtypes_df = dtypes_df.rename(columns = {0:'Columns', 1:'Pandas_dtype', 2:'Polars_dtype'})
dtypes_df


In [18]:
df_pd.head(3)

,item_id,status,created_at,sku,price,qty_ordered,grand_total,increment_id,category_name_1,sales_commission_code,...,payment_method,Working Date,BI Status,MV,Year,Month,Customer Since,M-Y,FY,Customer ID
0,211131,complete,07/01/2016,kreations_YI 06-L,1950.0,1,1950.0,100147443,Women's Fashion,\N,...,cod,07/01/2016,#REF!,"1,950",2016,7,2016-7,Jul-16,FY17,1.0
1,211133,canceled,07/01/2016,kcc_Buy 2 Frey Air Freshener & Get 1 Kasual Bo...,240.0,1,240.0,100147444,Beauty & Grooming,\N,...,cod,07/01/2016,Gross,240,2016,7,2016-7,Jul-16,FY17,2.0
2,211134,canceled,07/01/2016,Ego_UP0017-999-MR0,2450.0,1,2450.0,100147445,Women's Fashion,\N,...,cod,07/01/2016,Gross,"2,450",2016,7,2016-7,Jul-16,FY17,3.0


In [4]:
df_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 584524 entries, 0 to 584523
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   item_id                584524 non-null  int64  
 1   status                 584509 non-null  object 
 2   created_at             584524 non-null  object 
 3   sku                    584504 non-null  object 
 4   price                  584524 non-null  float64
 5   qty_ordered            584524 non-null  int64  
 6   grand_total            584524 non-null  float64
 7   increment_id           584524 non-null  object 
 8   category_name_1        584360 non-null  object 
 9   sales_commission_code  447346 non-null  object 
 10  discount_amount        584524 non-null  float64
 11  payment_method         584524 non-null  object 
 12  Working Date           584524 non-null  object 
 13  BI Status              584524 non-null  object 
 14   MV                    584524 non-nu

### FUNCTIONS

In [4]:
def df_cleaning(partition_df):
    """
    This function drops irrelevant columns and selected 
    rows from the partitions.
    Args:
        partition_df (_type_): _description_
    """
    col_to_drop = ['sales_commission_code', 'Working Date', 'M-Y']
    for cols in col_to_drop:
        if cols in partition_df.columns:
            partition_df = partition_df.drop(columns=cols)
        else:
            pass

    vals_dropped_df = partition_df.dropna()

    vals_dropped_df = vals_dropped_df[~(vals_dropped_df['category_name_1'] == r'\N')]
    vals_dropped_df = vals_dropped_df[~(vals_dropped_df['status'] == r'\N')]
    vals_dropped_df = vals_dropped_df[~(vals_dropped_df['BI Status'] == r'#REF!')]
    # vals_dropped_df = vals_dropped_df.reset_index()
    vals_dropped_df.reset_index(inplace=True)

    return vals_dropped_df



def df_transforming(data):
    """
    This function transforms the data
    Args:
        data (pd.DataFrame): Cleaned Dataframe

    Returns:
        pd.DataFrame: Transformed Data
    """
    data[' MV ' ] = (data[' MV '].replace(r',', '', regex=True))
    data[' MV ' ] = (data[' MV '].replace(r'-', 0, regex=True))
    data['Customer ID'] = data['Customer ID'].astype(int)
    data[' MV ' ] = pd.to_numeric(data[' MV '])#, errors='coerce')
    month_map = {1: 'January', 2: 'February', 3: 'March',
                4: 'April', 5: 'May', 6: 'June', 7: 'July',
                8: 'August', 9: 'September', 10: 'October', 
                11: 'November', 12: 'December'}
    data['Month'] = data['Month'].map(month_map)

    data = data.rename(columns={'sku': 'stock_keeping_unit', 'category_name_1': 'category_name',
                        ' MV ': 'market_value', 'BI Status': 'BI_status',
                        'Customer Since': 'customer_since', 'Customer ID': 'customer_id'})
    data = data.drop(columns='index')
    return data

In [5]:
clean_df = df_cleaning(df_pd)
trans_df = df_transforming(clean_df)

trans_df.info()
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 576480 entries, 0 to 576479
Data columns (total 18 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   item_id             576480 non-null  int64  
 1   status              576480 non-null  object 
 2   created_at          576480 non-null  object 
 3   stock_keeping_unit  576480 non-null  object 
 4   price               576480 non-null  float64
 5   qty_ordered         576480 non-null  int64  
 6   grand_total         576480 non-null  float64
 7   increment_id        576480 non-null  object 
 8   category_name       576480 non-null  object 
 9   discount_amount     576480 non-null  float64
 10  payment_method      576480 non-null  object 
 11  BI_status           576480 non-null  object 
 12  market_value        576480 non-null  int64  
 13  Year                576480 non-null  int64  
 14  Month               576480 non-null  object 
 15  customer_since      576480 non-nul

In [25]:
import numpy as np

"237983"

pd.set_option('display.max_rows', None)
# for row in trans_df[trans_df.market_value.isna()]:
#     if row == np.True_:
#         print(row.index)

ab = trans_df[trans_df.market_value.isna() == np.True_]
ac = ab.index

ad = ac.tolist()
print(len(ad))


trans_df.iloc[ad]

df_pd[df_pd['item_id'] == 218538]

2203


,item_id,status,created_at,sku,price,qty_ordered,grand_total,increment_id,category_name_1,sales_commission_code,...,payment_method,Working Date,BI Status,MV,Year,Month,Customer Since,M-Y,FY,Customer ID
6598,218538,canceled,7/26/2016,"west point_Deluxe Juicer, Blender & Grinder - ...",0.0,1,0.0,100152813,Appliances,\N,...,cod,7/26/2016,Gross,-,2016,7,2016-7,Jul-16,FY17,1961.0


Convert 
- MV columns to float
- Month integer to String and name of month in reality.
- Customer Id to Integer


Drop 
- Working Date and created_at are the same thing. 
- Customer_since and M-Y are the same thing
    + Drop Working date, M-Y

In [30]:
clean_df[ab]

ValueError: Boolean array expected for the condition, not object

In [ ]:
# # ad = clean_df['Working Date'] == clean_df['created_at']
# af = clean_df['Customer Since'] == clean_df['M-Y']
# ac = pd.Series(ac)
# ac
# ac.nunique()

KeyError: 'M-Y'

In [12]:
ab.columns

Index(['index', 'item_id', 'status', 'created_at', 'sku', 'price',
       'qty_ordered', 'grand_total', 'increment_id', 'category_name_1',
       'discount_amount', 'payment_method', 'Working Date', 'BI Status',
       ' MV ', 'Year', 'Month', 'Customer Since', 'M-Y', 'FY', 'Customer ID'],
      dtype='object')

Database Tables

Customers
- customer_id
- customer_since

Products
- 
- sku
- category_name_1

Orders
- item_id
- status
- created_at
- price
- qty_ordered
- payment method
- Month

sales
- grand_total
- BI Status
- MV
- discount_amount
- FY


In [23]:
ab.tail(5)

,index,item_id,status,created_at,sku,price,qty_ordered,grand_total,increment_id,category_name_1,discount_amount,payment_method,BI Status,MV,Year,Month,Customer Since,FY,Customer ID
576475,584519,905204,cod,8/28/2018,WOFSCE5AE00357AECDE,699.0,1,849.0,100562385,Women's Fashion,0.0,cod,Valid,699,2018,8,2018-8,FY19,115320.0
576476,584520,905205,processing,8/28/2018,MATHUA5AF70A7D1E50A,35599.0,1,35899.0,100562386,Mobiles & Tablets,0.0,bankalfalah,Gross,"35,599",2018,8,2018-8,FY19,115326.0
576477,584521,905206,processing,8/28/2018,MATSAM5B6D7208C6D30,129999.0,2,652178.0,100562387,Mobiles & Tablets,0.0,bankalfalah,Gross,"259,998",2018,8,2018-7,FY19,113474.0
576478,584522,905207,processing,8/28/2018,MATSAM5B1509B4696EA,87300.0,2,652178.0,100562387,Mobiles & Tablets,0.0,bankalfalah,Gross,"174,600",2018,8,2018-7,FY19,113474.0
576479,584523,905208,processing,8/28/2018,MATSAM5B10F91A9B6AB,108640.0,2,652178.0,100562387,Mobiles & Tablets,0.0,bankalfalah,Gross,"217,280",2018,8,2018-7,FY19,113474.0


In [79]:
584523-576481

8042

In [ ]:
den = df_pl.drop_nulls()
# den.describe()

dan = df_pd.dropna()
# dan.describe()
# Nulls dropped

dan.info()
df_pd # NAs present


### Checking Partionable Columns with Null Values Removed

In [ ]:
dict_list = []
for col in df_pd.columns:
    ab = df_pd[col].nunique()
    dict_list.append(ab)

for i in dict_list:
    print(i)





584524
16
789
84889
9121
74
36829
408785
16
7224
28058
18
789
4
9720
3
12
26
26
3
115326


### Checking Partionable Columns with Null Values Present

In [22]:
dict_list = []
for col in df_pd.columns:
    ab = df_pd[col].value_counts()
    dict_list.append(ab)

for i in dict_list:
    print(i)

# Null values present

item_id
905208    1
211131    1
211133    1
211134    1
211135    1
         ..
211149    1
211147    1
211146    1
211145    1
211144    1
Name: count, Length: 584524, dtype: int64
status
complete          233685
canceled          201249
received           77290
order_refunded     59529
refund              8050
cod                 2859
paid                1159
closed               494
payment_review        57
pending               48
processing            33
holded                31
fraud                 10
pending_paypal         7
\N                     4
exchange               4
Name: count, dtype: int64
created_at
11/25/2016    15169
11/17/2017    13698
11/24/2017    13191
5/19/2017     11511
11/23/2016     8478
              ...  
8/22/2018        92
9/14/2016        83
07/06/2016       72
9/13/2016        52
07/07/2016       51
Name: count, Length: 789, dtype: int64
sku
MATSAM59DB75ADB2F80              3775
Al Muhafiz Sohan Halwa Almond    2258
emart_00-7                       20

In [ ]:
for col in df_pd.columns:
    print(df_pd[df_pd[col] == r"NaN"].count())

item_id                  0
status                   0
created_at               0
sku                      0
price                    0
qty_ordered              0
grand_total              0
increment_id             0
category_name_1          0
sales_commission_code    0
discount_amount          0
payment_method           0
Working Date             0
BI Status                0
 MV                      0
Year                     0
Month                    0
Customer Since           0
M-Y                      0
FY                       0
Customer ID              0
dtype: int64
item_id                  0
status                   0
created_at               0
sku                      0
price                    0
qty_ordered              0
grand_total              0
increment_id             0
category_name_1          0
sales_commission_code    0
discount_amount          0
payment_method           0
Working Date             0
BI Status                0
 MV                      0
Year           

In [45]:
df_pd['sales_commission_code'].replace(r"\N", 'Unknown')

0             Unknown
1             Unknown
2             Unknown
3         R-FSD-52352
4             Unknown
             ...     
584519            NaN
584520            NaN
584521            NaN
584522            NaN
584523            NaN
Name: sales_commission_code, Length: 584524, dtype: object

0                  \N
1                  \N
2                  \N
3         R-FSD-52352
4                  \N
             ...     
584519            NaN
584520            NaN
584521            NaN
584522            NaN
584523            NaN
Name: sales_commission_code, Length: 584524, dtype: object